
# Overview
### Model training is an iterative and experimental process.

![image-alt-text](https://synapseaisolutionsa.blob.core.windows.net/public/imgs/flaml_modeladaptation.png)

In [ ]:
%pip install "flaml[synapse]@https://automlsaeastus.blob.core.windows.net/releases/FLAML-latest-py3-none-any.whl"

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, -1, Finished, Available)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 1.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.1/66.1 kB 1.6 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... - done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 5.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 kB 38.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 37.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.7/78.7 kB 40.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 28.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.2/147.2 kB 53.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 57.6 MB/s eta 0:00:00
  Created wheel for joblibspark: filename=joblibspark-0.5.2-py3-none-any.whl size=15260 sha256=8d19a9a9192200ce12937609d5de2baed4a5a6843d4e6a204dcb43876d003c88
  Stored in directory: /home/trusted-ser

In [ ]:
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 9, Finished, Available)

# Demo overview
|  | | | | |
|-----|-----|--------|--------|--------|
|![synapse](https://microsoft.github.io/SynapseML/img/logo.svg)| <img src="https://www.microsoft.com/en-us/research/uploads/prod/2020/02/flaml-1024x406.png" alt="drawing" width="200"/> | ![image-alt-text](https://th.bing.com/th/id/OIP.5aNnFabBKoYIYhoTrNc_CAHaHa?w=174&h=180&c=7&r=0&o=5&pid=1.7)| 


<style>
td, th {
   border: none!important;
}
</style>


#### 1. **Hyperparameter Tuning**: Given a dataset, model task (e.g. classification), model type (e.g. LightGBM), and range of parameters (e.g. number of iterations, number of leaves, max_depth), help me find the best parameters for a given metric (e.g. accuracy).
#### 2. **AutoML**: Given a dataset & model task, help me find the best model. 

![image-alt-text](https://synapseaisolutionsa.blob.core.windows.net/public/imgs/flaml_scenario.png)

## 2. Load data and preprocess

In [ ]:
df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(
        "wasbs://publicwasb@mmlspark.blob.core.windows.net/company_bankruptcy_prediction_data.csv"
    )
)
# print dataset size
print("records read: " + str(df.count()))

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 10, Finished, Available)

records read: 6819


In [ ]:
display(df)

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 11, Finished, Available)

SynapseWidget(Synapse.DataFrame, f0efdb50-9275-4b31-92b5-d0ac89b191e2)

### Featurize the dataset 

In [ ]:
train_raw, test_raw = df.randomSplit([0.8, 0.2], seed=41)

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 12, Finished, Available)

In [ ]:
from pyspark.ml.feature import VectorAssembler

feature_cols = df.columns[1:]
featurizer = VectorAssembler(inputCols=feature_cols, outputCol="features")
train_data = featurizer.transform(train_raw)["Bankrupt?", "features"]
test_data = featurizer.transform(test_raw)["Bankrupt?", "features"]

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 13, Finished, Available)

## 3. Train the Default SynapseML Model

In [ ]:
import mlflow
mlflow.set_experiment("flaml_tune_demo")
mlflow.autolog(exclusive=False)

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 14, Finished, Available)

2023/07/13 20:59:03 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.ml.


In [ ]:
def predict(model, test_data=test_data):

    predictions = model.transform(test_data)
    
    metrics = ComputeModelStatistics(
        evaluationMetric="classification",
        labelCol="Bankrupt?",
        scoredLabelsCol="prediction",
    ).transform(predictions)
    return metrics


StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 15, Finished, Available)

In [ ]:
from synapse.ml.lightgbm import LightGBMClassifier
from synapse.ml.train import ComputeModelStatistics

with mlflow.start_run(run_name="tune_default") as run:
    model = LightGBMClassifier(objective="binary", featuresCol="features", labelCol="Bankrupt?", isUnbalance=True)
    model = model.fit(train_data)

    # Generate predictions and log metrics
    default_metrics = predict(model)
    
    mlflow.log_metrics({"accuracy": default_metrics.first()['accuracy'], "AUC": default_metrics.first()['AUC']})

    default_metrics.show()

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 16, Finished, Available)

2023/07/13 20:59:09 WARNING mlflow.pyspark.ml: Model inputs contain unsupported Spark data types: [StructField('features', VectorUDT(), True)]. Model signature is not logged.
2023/07/13 20:59:16 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmp43xufw80/model, flavor: spark), fall back to return ['pyspark==3.3.1']. Set logging level to DEBUG to see the full traceback.
2023/07/13 20:59:16 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/home/trusted-service-user/cluster-env/trident_env/lib/python3.10/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils."


+---------------+--------------------+-----------------+------------------+-------------------+------------------+
|evaluation_type|    confusion_matrix|         accuracy|         precision|             recall|               AUC|
+---------------+--------------------+-----------------+------------------+-------------------+------------------+
| Classification|1250.0  23.0  \n3...|0.958997722095672|0.3611111111111111|0.29545454545454547|0.6386934942512319|
+---------------+--------------------+-----------------+------------------+-------------------+------------------+



## 4. Tune the model with FLAML Tune

In [ ]:
def train(lambdaL1, learningRate, numLeaves, numIterations, train_data=train_data, val_data=test_data):
    """
    This train() function:
     - takes hyperparameters as inputs (for tuning later)
     - returns the AUC score on the validation dataset

    Wrapping code as a function makes it easier to reuse the code later for tuning.
    """
    lgc = LightGBMClassifier(
        objective="binary",
        lambdaL1=lambdaL1,
        learningRate=learningRate,
        numLeaves=numLeaves,
        labelCol="Bankrupt?",
        numIterations=numIterations,
        isUnbalance=True,
        featuresCol="features",
    )
    model = lgc.fit(train_data)
    # Define an evaluation metric and evaluate the model on the validation dataset.
    eval_metric = predict(model, val_data)
    eval_metric = eval_metric.toPandas()['AUC'][0]
    return model, eval_metric

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 17, Finished, Available)

In [ ]:
import flaml
import time

# define the search space
params = {
    "lambdaL1": flaml.tune.uniform(0.001, 1),
    "learningRate": flaml.tune.uniform(0.001, 1),
    "numLeaves": flaml.tune.randint(30, 100),
    "numIterations": flaml.tune.randint(100, 300),
}

# define the tune function
def flaml_tune(config):
    _, metric = train(**config)
    return {"auc": metric}

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 18, Finished, Available)

2023/07/13 20:59:32 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2023/07/13 20:59:33 INFO mlflow.tracking.fluent: Autologging successfully enabled for lightgbm.
2023/07/13 20:59:33 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.


In [ ]:
with mlflow.start_run(nested=True, run_name="Child Run: "):
    analysis = flaml.tune.run(
        flaml_tune,
        params,
        time_budget_s=60,
        num_samples=100,
        metric="auc",
        mode="max",
        verbose=2,
        force_cancel=True,
        )

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 19, Finished, Available)

[flaml.fabric._telemetry: 07-13 20:59:36] {24} INFO - log_telemetry: flaml-tune
log_telemetry: flaml-tune
[flaml.tune.tune: 07-13 20:59:37] {562} INFO - Using search algorithm BlendSearch.
No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune
You passed a `space` parameter to OptunaSearch that contained unresolved search space definitions. OptunaSearch should however be instantiated with fully configured search spaces only. To use Ray Tune's automatic search space conversion, pass the space definition as part of the `config` argument to `tune.run()` instead.
[flaml.tune.tune: 07-13 20:59:37] {852} INFO - trial 1 config: {'lambdaL1': 0.09833464080607023, 'learningRate': 0.64761881525086, 'numLeaves': 30, 'numIterations': 172}
[flaml.tune.tune: 07-13 20:59:47] {852} INF

[I 2023-07-13 20:59:37,559] A new study created in memory with name: optuna
Time exceeded, canceled jobs


Best config and metric on validation data

In [ ]:
tune_config = analysis.best_config
tune_metrics_val = analysis.best_result
print("Best config: ", tune_config)
print("Best metrics on validation data: ", tune_metrics_val)

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 20, Finished, Available)

Best config:  {'lambdaL1': 0.09833464080607023, 'learningRate': 0.64761881525086, 'numLeaves': 30, 'numIterations': 172}
Best metrics on validation data:  {'auc': 0.805291723202171, 'training_iteration': 0, 'config': {'lambdaL1': 0.09833464080607023, 'learningRate': 0.64761881525086, 'numLeaves': 30, 'numIterations': 172}, 'config/lambdaL1': 0.09833464080607023, 'config/learningRate': 0.64761881525086, 'config/numLeaves': 30, 'config/numIterations': 172, 'experiment_tag': 'exp', 'time_total_s': 9.833300113677979}


Retrain model on whole train_data and check metrics on test_data

In [ ]:
tune_model, tune_metrics = train(train_data=train_data, val_data=test_data, **tune_config)
tune_metrics = predict(tune_model)
tune_metrics.show()

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 21, Finished, Available)

+---------------+--------------------+------------------+-------------------+------------------+-----------------+
|evaluation_type|    confusion_matrix|          accuracy|          precision|            recall|              AUC|
+---------------+--------------------+------------------+-------------------+------------------+-----------------+
| Classification|893.0  380.0  \n4...|0.7084282460136674|0.09523809523809523|0.9090909090909091|0.805291723202171|
+---------------+--------------------+------------------+-------------------+------------------+-----------------+



## 4. Use AutoML to find the best model


In [ ]:
''' import AutoML class from the FLAML package '''
from flaml import AutoML
from flaml.automl.spark.utils import to_pandas_on_spark
automl = AutoML()

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 34, Finished, Available)

In [ ]:
import mlflow
mlflow.autolog()
mlflow.set_experiment("automl_classification_demo")

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 35, Finished, Available)

2023/07/13 21:13:24 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.
2023/07/13 21:13:24 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2023/07/13 21:13:24 INFO mlflow.tracking.fluent: Autologging successfully enabled for lightgbm.
2023/07/13 21:13:24 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.ml.


<Experiment: artifact_location='', creation_time=1686600263597, experiment_id='3b8192d6-f9c6-46c6-84b2-c10deb3e8566', last_update_time=None, lifecycle_stage='active', name='automl_classification_demo', tags={}>

In [ ]:
import os
settings = {
    "time_budget": 120,  # total running time in seconds
    "metric": 'roc_auc',
    "task": 'classification',  # task type
    "log_file_name": 'flaml_experiment.log',  # flaml log file
    "seed": 42,    # random seed
    "force_cancel": True,  # force stop training once time_budget is used up
    "mlflow_exp_name": "automl_classification_demo"
}

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 36, Finished, Available)

In [ ]:
df = to_pandas_on_spark(train_data)
type(df)

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 37, Finished, Available)

pyspark.pandas.frame.DataFrame

In [ ]:
'''The main flaml automl API'''

with mlflow.start_run(nested=True):
    automl.fit(dataframe=df, label='Bankrupt?', isUnbalance=True, **settings)

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 38, Finished, Available)

[flaml.automl.logger: 07-13 21:13:54] {1707} INFO - task = classification
[flaml.automl.logger: 07-13 21:13:54] {1714} INFO - Data split method: stratified
[flaml.automl.logger: 07-13 21:13:54] {1717} INFO - Evaluation method: cv
[flaml.automl.logger: 07-13 21:13:54] {1815} INFO - Minimizing error metric: 1-roc_auc
[flaml.automl.logger: 07-13 21:13:55] {1927} INFO - List of ML learners in AutoML Run: ['lgbm_spark', 'rf_spark']
[flaml.automl.logger: 07-13 21:13:55] {2219} INFO - iteration 0, current learner lgbm_spark
[flaml.automl.logger: 07-13 21:14:32] {2347} INFO - Estimated sufficient time budget=369594s. Estimated necessary time budget=370s.
[flaml.automl.logger: 07-13 21:14:32] {2396} INFO -  at 38.6s,	estimator lgbm_spark's best error=0.1077,	best estimator lgbm_spark's best error=0.1077
[flaml.automl.logger: 07-13 21:14:32] {2219} INFO - iteration 1, current learner lgbm_spark
[flaml.automl.logger: 07-13 21:15:08] {2396} INFO -  at 74.8s,	estimator lgbm_spark's best error=0.096

### Best model and metric

In [ ]:
''' retrieve best config'''
print('Best hyperparmeter config:', automl.best_config)
print('Best roc_auc on validation data: {0:.4g}'.format(1-automl.best_loss))
print('Training duration of best run: {0:.4g} s'.format(automl.best_config_train_time))

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 39, Finished, Available)

Best hyperparmeter config: {'numTrees': 4, 'featureSubsetStrategy': 1.0, 'maxDepth': 4, 'impurity': 'gini'}
Best roc_auc on validation data: 0.9407
Training duration of best run: 7.595 s


In [ ]:
automl_metrics = predict(automl.model.estimator)
automl_metrics.show()

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 40, Finished, Available)

+---------------+--------------------+------------------+---------+-------------------+------------------+
|evaluation_type|    confusion_matrix|          accuracy|precision|             recall|               AUC|
+---------------+--------------------+------------------+---------+-------------------+------------------+
| Classification|1268.0  5.0  \n39...|0.9665907365223994|      0.5|0.11363636363636363|0.5548543169320859|
+---------------+--------------------+------------------+---------+-------------------+------------------+



## 5. Use Apache Spark to Parallelize AutoML trials

In [ ]:
import mlflow
mlflow.autolog()
mlflow.set_experiment("automl_spark_demo")

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 29, Finished, Available)

2023/07/13 21:05:54 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.
2023/07/13 21:05:54 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2023/07/13 21:05:54 INFO mlflow.tracking.fluent: Autologging successfully enabled for lightgbm.
2023/07/13 21:05:54 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.ml.


<Experiment: artifact_location='', creation_time=1686600858470, experiment_id='016b1109-7657-4167-a86b-b3dca08f97dd', last_update_time=None, lifecycle_stage='active', name='automl_spark_demo', tags={}>

In [ ]:
settings = {
    "time_budget": 120,  # total running time in seconds
    "metric": 'roc_auc',  # primary metrics for regression can be chosen from: ['mae','mse','r2','rmse','mape']
    "task": 'classification',  # task type    
    "seed": 7654321,    # random seed
    "use_spark": True,
    "n_concurrent_trials": 3,
    "force_cancel": True,
    "mlflow_exp_name": "automl_spark_demo"

}

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 30, Finished, Available)

In [ ]:
pandas_df = train_raw.toPandas()
pandas_df.head()

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 31, Finished, Available)

,Bankrupt?,ROA(C) before interest and depreciation before interest,ROA(A) before interest and % after tax,ROA(B) before interest and depreciation after tax,Operating Gross Margin,Realized Sales Gross Margin,Operating Profit Rate,Pre-tax net Interest Rate,After-tax net Interest Rate,Non-industry income and expenditure/revenue,...,Net Income to Total Assets,Total assets to GNP price,No-credit Interval,Gross Profit to Sales,Net Income to Stockholder's Equity,Liability to Equity,Degree of Financial Leverage (DFL),Interest Coverage Ratio (Interest expense to EBIT),Net Income Flag,Equity to Liability
0,0,0.0828,0.0693,0.0884,0.6468,0.6468,0.9971,0.7958,0.8078,0.3047,...,0.0000,0.000000e+00,0.6237,0.6468,0.7483,0.2847,0.0268,0.5652,1.0,0.0199
1,0,0.1606,0.1788,0.1832,0.5897,0.5897,0.9986,0.7969,0.8088,0.3034,...,0.5917,4.370000e+09,0.6236,0.5897,0.8023,0.2947,0.0268,0.5651,1.0,0.0151
2,0,0.2040,0.2638,0.2598,0.4483,0.4483,0.9959,0.7937,0.8063,0.3034,...,0.6816,3.000000e-04,0.6221,0.4483,0.8117,0.3038,0.0268,0.5651,1.0,0.0136
3,0,0.2170,0.1881,0.2451,0.5992,0.5992,0.9962,0.7940,0.8061,0.3034,...,0.6196,1.100000e-03,0.6236,0.5992,0.6346,0.4359,0.0268,0.5650,1.0,0.0108
4,0,0.2314,0.1628,0.2068,0.6001,0.6001,0.9988,0.7960,0.8078,0.3015,...,0.5269,3.000000e-04,0.6241,0.6001,0.7985,0.2903,0.0268,0.5651,1.0,0.0164


In [ ]:
'''The main flaml automl API'''
with mlflow.start_run(nested=True):
    automl.fit(dataframe=pandas_df, label='Bankrupt?', **settings)

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 32, Finished, Available)

[flaml.automl.logger: 07-13 21:05:59] {1707} INFO - task = classification
[flaml.automl.logger: 07-13 21:05:59] {1714} INFO - Data split method: stratified
[flaml.automl.logger: 07-13 21:05:59] {1717} INFO - Evaluation method: holdout
[flaml.automl.logger: 07-13 21:05:59] {1815} INFO - Minimizing error metric: 1-roc_auc
[flaml.automl.logger: 07-13 21:05:59] {1927} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'lrl1']
[flaml.tune.tune: 07-13 21:05:59] {762} INFO - Number of trials: 1/1000000, 1 RUNNING, 0 TERMINATED
[flaml.tune.tune: 07-13 21:06:09] {785} INFO - Brief result: {'pred_time': 4.134316375290138e-06, 'wall_clock_time': 10.414557456970215, 'metric_for_logging': {'pred_time': 4.134316375290138e-06}, 'val_loss': 0.04636121259998027, 'trained_estimator': <flaml.automl.model.LGBMEstimator object at 0x7f5b892ddc00>}
[flaml.tune.tune: 07-13 21:06:09] {762} INFO - Number of trials: 2/1000000, 1 RUNNING, 1 TERMINATED
[flaml.tune.t

[I 2023-07-13 21:05:59,304] A new study created in memory with name: optuna
[I 2023-07-13 21:05:59,595] A new study created in memory with name: optuna
Time exceeded, canceled jobs


In [ ]:
''' retrieve best config'''
print('Best hyperparmeter config:', automl.best_config)
print('Best roc_auc on validation data: {0:.4g}'.format(1-automl.best_loss))
print('Training duration of best run: {0:.4g} s'.format(automl.best_config_train_time))

StatementMeta(, 05b98e4b-8785-495f-ae6a-9a8b19ce7451, 33, Finished, Available)

Best hyperparmeter config: {'n_estimators': 4, 'num_leaves': 4, 'min_child_samples': 12, 'learning_rate': 0.12219010506857894, 'log_max_bin': 8, 'colsample_bytree': 0.8752902632453101, 'reg_alpha': 0.005456783453304973, 'reg_lambda': 1.1612933757574404}
Best roc_auc on validation data: 0.9539
Training duration of best run: 3.76 s
